# 🚗 Car Price Prediction Using Machine Learning
### **OASIS INFOBYTE Data Science Internship — Task 3**

**Author:** Senior Machine Learning Engineer & Data Scientist  
**Dataset:** CarDekho Used Car Details Dataset ( = 4,340$ raw listings)  
**Objective:** Develop an end-to-end regression system to predict used-car selling prices in India with leakage-free scikit-learn pipelines, model explainability, and interactive dashboard deployment.


---
## Phase 1: Environment Setup & Data Ingestion
We load the authentic CarDekho dataset (car_details.csv) containing 4,340 used car listings.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Configure paths
raw_path = os.path.join("..", "data", "raw", "car_details.csv")
df_raw = pd.read_csv(raw_path)
print(f"Raw dataset shape: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns.")
df_raw.head()

---
## Phase 2: Data Inspection & Missing Value Analysis

In [ ]:
print("--- Dataset Information ---")
df_raw.info()
print("\n--- Missing Values Count ---")
print(df_raw.isnull().sum())
print(f"\nDuplicate rows detected: {df_raw.duplicated().sum()}")

---
## Phase 3: Data Cleaning & Deduplication
We remove 763 exact duplicate listings to prevent train/test data leakage, strip whitespace, and standardize brand names.

In [ ]:
df = df_raw.drop_duplicates().reset_index(drop=True)
print(f"Unique cleaned records: {len(df)}")

# Standardize brand extraction
def extract_clean_brand(name):
    toks = str(name).split()
    if not toks:
        return "Unknown"
    if toks[0] == "OpelCorsa":
        return "Opel"
    if toks[0] == "Land" and len(toks) > 1 and toks[1] == "Rover":
        return "Land Rover"
    return toks[0]

df["brand"] = df["name"].apply(extract_clean_brand)
REFERENCE_YEAR = 2020
df["car_age"] = REFERENCE_YEAR - df["year"]
df[["name", "brand", "year", "car_age", "selling_price", "km_driven", "fuel", "transmission"]].head()

---
## Phase 4: Exploratory Data Analysis (EDA)
We evaluate market dynamics: price distribution, depreciation curves, fuel types, and transmission premiums.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df["selling_price"] / 1e5, kde=True, ax=ax1, color="#059669", bins=35)
ax1.set_title("Selling Price Distribution (in Lakhs INR)", fontweight="bold")
ax1.set_xlabel("Price (₹ in Lakhs)")

sns.regplot(data=df, x="car_age", y=df["selling_price"] / 1e5, order=2, ax=ax2,
            scatter_kws={"alpha": 0.35, "color": "#059669", "s": 20},
            line_kws={"color": "#dc2626", "lw": 2})
ax2.set_title("Depreciation: Price vs. Car Age", fontweight="bold")
ax2.set_xlabel("Car Age (Years)")
ax2.set_ylabel("Price (₹ in Lakhs)")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df, x="transmission", y=df["selling_price"] / 1e5, palette=["#059669", "#2563eb"], ax=ax)
ax.set_title("Selling Price by Transmission Type (Automatic vs. Manual)", fontweight="bold")
ax.set_xlabel("Transmission")
ax.set_ylabel("Selling Price (₹ in Lakhs)")
plt.show()

---
## Phase 5: Machine Learning Pipeline & Cross-Validation
We set up a leakage-safe ColumnTransformer with StandardScaler on numericals and OneHotEncoder(handle_unknown="ignore") on categoricals. We benchmark 4 regressors with log-transformed targets using 5-fold cross-validation.

In [ ]:
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

features = ["brand", "car_age", "km_driven", "fuel", "seller_type", "transmission", "owner"]
X = df[features]
y = df["selling_price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

num_cols = ["car_age", "km_driven"]
cat_cols = ["brand", "fuel", "seller_type", "transmission", "owner"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_cols)
])

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=2.0),
    "Random Forest": RandomForestRegressor(n_estimators=180, max_depth=16, min_samples_split=4, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=180, learning_rate=0.08, max_depth=5, random_state=42)
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
for name, reg in models.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("regressor", reg)])
    t_model = TransformedTargetRegressor(regressor=pipe, func=np.log1p, inverse_func=np.expm1)
    cv_res = cross_validate(t_model, X_train, y_train, cv=kf, scoring={"r2": "r2", "mae": "neg_mean_absolute_error"}, n_jobs=-1)
    print(f"{name:20s} | 5-Fold CV R2: {cv_res['test_r2'].mean():.4f} (+/- {cv_res['test_r2'].std():.4f}) | CV MAE: INR {-cv_res['test_mae'].mean():,.0f}")

---
## Phase 6: Held-Out Test Evaluation & Diagnostics
We fit the best-performing model on the entire training set and evaluate generalization on the held-out test set.

In [ ]:
best_pipe = Pipeline([("preprocessor", preprocessor), ("regressor", LinearRegression())])
best_model = TransformedTargetRegressor(regressor=best_pipe, func=np.log1p, inverse_func=np.expm1)
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Held-Out Test Set Results:")
print(f"  R² Score : {r2:.4f}")
print(f"  Test MAE : ₹{mae:,.2f}")
print(f"  Test RMSE: ₹{rmse:,.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(y_test / 1e5, y_pred / 1e5, alpha=0.55, color="#059669", edgecolors="white", s=32)
max_val = max((y_test/1e5).max(), (y_pred/1e5).max()) * 1.05
ax.plot([0, max_val], [0, max_val], color="#dc2626", linestyle="--", lw=2, label="Ideal (y = x)")
ax.set_title("Actual vs. Predicted Selling Price (Held-Out Test Set)", fontweight="bold")
ax.set_xlabel("Actual Price (₹ in Lakhs)")
ax.set_ylabel("Predicted Price (₹ in Lakhs)")
ax.legend()
plt.show()

---
## Phase 7: Production Inference Verification
We test model inference on a sample user input to confirm real-time valuation capability.

In [ ]:
sample_input = pd.DataFrame([{
    "brand": "Hyundai",
    "car_age": 4.0,
    "km_driven": 35000.0,
    "fuel": "Petrol",
    "seller_type": "Individual",
    "transmission": "Manual",
    "owner": "First Owner"
}])
pred_val = best_model.predict(sample_input)[0]
print(f"Sample Vehicle Valuation: ₹{pred_val:,.0f} (₹{pred_val/1e5:.2f} Lakh)")